# Primary Auditor LLM

This notebook implements the primary scoring model for the project. It works sentence by sentence, treating each sentence as the evidence span, and then aggregates verified sentence scores into document-level scores.

## 1. Install dependencies

In [1]:
!pip -q install transformers datasets accelerate evaluate scikit-learn pandas numpy pypdf

## 2. Workspace and BASIL download

In [2]:
from pathlib import Path
import urllib.request
import zipfile

ROOT = Path('/content') if Path('/content').exists() else (Path.cwd() / 'basil_workspace')
ROOT.mkdir(parents=True, exist_ok=True)

ARCHIVE_URL = 'https://github.com/marshallwhiteorg/emnlp19-media-bias/archive/refs/heads/master.zip'
ARCHIVE_PATH = ROOT / 'basil_source.zip'
EXTRACT_DIR = ROOT / 'basil_source'
COMBINED_ZIP = EXTRACT_DIR / 'emnlp19-media-bias-master' / 'emnlp19-BASIL.zip'
COMBINED_DIR = ROOT / 'emnlp19-BASIL'
DATASET_DIR = COMBINED_DIR / 'data'

if not ARCHIVE_PATH.exists():
    urllib.request.urlretrieve(ARCHIVE_URL, ARCHIVE_PATH)
if not EXTRACT_DIR.exists():
    with zipfile.ZipFile(ARCHIVE_PATH) as zf:
        zf.extractall(EXTRACT_DIR)
if not COMBINED_DIR.exists():
    with zipfile.ZipFile(COMBINED_ZIP) as zf:
        zf.extractall(ROOT)

print('ROOT:', ROOT)
print('DATASET_DIR:', DATASET_DIR)
print('JSON files:', len(list(DATASET_DIR.glob('*.json'))))

ROOT: /home/rwb6928/basil_workspace
DATASET_DIR: /home/rwb6928/basil_workspace/emnlp19-BASIL/data
JSON files: 300


## 3. BASIL sentence dataset

In [3]:
from __future__ import annotations

import json
from typing import Any

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit


def load_basil_sentences(dataset_dir: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for path in sorted(dataset_dir.glob('*.json')):
        article = json.loads(path.read_text())
        article_id = path.stem
        event_id, source = article_id.split('_', 1)
        sentence_count = len(article['body'])
        for sentence_obj in article['body']:
            sentence = (sentence_obj.get('sentence') or '').strip()
            annotations = sentence_obj.get('annotations', [])
            bias_types = sorted({a.get('bias', '').strip().lower() for a in annotations if a.get('bias')})
            rows.append({
                'example_id': f"{article_id}::{sentence_obj['sentence-index']}",
                'event_id': event_id,
                'article_id': article_id,
                'source': source.lower(),
                'date': article.get('date'),
                'title': article.get('title'),
                'url': article.get('url'),
                'main_event': article.get('main-event'),
                'sentence_index': sentence_obj['sentence-index'],
                'sentence_count': sentence_count,
                'sentence_text': sentence,
                'label': int(bool(annotations)),
                'gold_annotation_count': len(annotations),
                'gold_bias_types': bias_types,
            })
    frame = pd.DataFrame(rows)
    frame['sentence_text'] = frame['sentence_text'].fillna('')
    return frame


def split_sentence_frame(frame: pd.DataFrame, test_size: float = 0.2, random_state: int = 42):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(frame, frame['label'], groups=frame['event_id']))
    train_frame = frame.iloc[train_idx].reset_index(drop=True)
    test_frame = frame.iloc[test_idx].reset_index(drop=True)
    return train_frame, test_frame


frame = load_basil_sentences(DATASET_DIR)
train_frame, test_frame = split_sentence_frame(frame)
print('train size', len(train_frame), 'test size', len(test_frame))
print('train positive rate', round(train_frame['label'].mean(), 4))
print('test positive rate', round(test_frame['label'].mean(), 4))

train size 6327 test size 1657
train positive rate 0.2036
test positive rate 0.2034


## 4. Proposal math helpers

In [4]:
from collections import defaultdict

def clamp01(value):
    return max(0.0, min(1.0, float(value)))

def discourse_position_weight(sentence_index, sentence_count, headline_weight=1.6, lede_weight=1.3, closing_weight=1.15, body_weight=1.0):
    if sentence_count <= 0:
        return body_weight
    if sentence_index == 0:
        return headline_weight
    if sentence_index <= min(2, sentence_count - 1):
        return lede_weight
    if sentence_index >= max(0, sentence_count - 2):
        return closing_weight
    return body_weight

def normalized_type_weights(label, sentence_text):
    text = sentence_text.lower()
    lexical_markers = ['outrageous', 'dishonest', 'radical', 'piggy bank', 'deranged', 'treasured entitlement']
    if label != 'biased':
        return {'lexical': 0.0, 'informational': 0.0}, ['none']
    if any(marker in text for marker in lexical_markers):
        return {'lexical': 0.65, 'informational': 0.35}, ['lexical_bias', 'sentiment_skew']
    return {'lexical': 0.35, 'informational': 0.65}, ['informational_bias', 'framing_bias']

def sentence_bias_scores(confidence, label, sentence_text, rationale=''):
    weights, bias_types = normalized_type_weights(label, sentence_text)
    active = 1.0 if label == 'biased' else 0.0
    bias_strength = clamp01(confidence) * active
    return {
        'bias_types': bias_types,
        's_lex': round(bias_strength * weights['lexical'], 6),
        's_inf': round(bias_strength * weights['informational'], 6),
        'rationale': rationale,
    }

def scalar_bias_score(scores, alpha=0.5, beta=0.5):
    return round(clamp01(alpha * scores['s_lex'] + beta * scores['s_inf']), 6)

def aggregate_article_scores(records, alpha=0.5, beta=0.5):
    weighted_lex = weighted_inf = total_weight = 0.0
    for record in records:
        weight = discourse_position_weight(int(record['sentence_index']), int(record['sentence_count']))
        weighted_lex += weight * float(record['judgment']['math_scores']['s_lex'])
        weighted_inf += weight * float(record['judgment']['math_scores']['s_inf'])
        total_weight += weight
    if total_weight == 0:
        return {'s_lex': 0.0, 's_inf': 0.0, 'document_bias_score': 0.0}
    s_lex = weighted_lex / total_weight
    s_inf = weighted_inf / total_weight
    return {
        's_lex': round(s_lex, 6),
        's_inf': round(s_inf, 6),
        'document_bias_score': round(clamp01(alpha * s_lex + beta * s_inf), 6),
    }


## 5. Train DistilBERT sentence scorer

In [ ]:
import numpy as np
import evaluate
from datasets import Dataset, DatasetDict
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

dataset_dict = DatasetDict({
    'train': Dataset.from_pandas(train_frame, preserve_index=False),
    'test': Dataset.from_pandas(test_frame, preserve_index=False),
})

def tokenize_batch(batch):
    return tokenizer(batch['sentence_text'], truncation=True, padding='max_length', max_length=256)

tokenized = dataset_dict.map(tokenize_batch, batched=True, batch_size=32, load_from_cache_file=False)
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

accuracy_metric = evaluate.load('accuracy')
precision_metric = evaluate.load('precision')
recall_metric = evaluate.load('recall')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    metrics = {}
    metrics.update(accuracy_metric.compute(predictions=preds, references=labels))
    metrics.update(precision_metric.compute(predictions=preds, references=labels, zero_division=0))
    metrics.update(recall_metric.compute(predictions=preds, references=labels, zero_division=0))
    metrics.update(f1_metric.compute(predictions=preds, references=labels))
    return metrics

OUTPUT_DIR = ROOT / 'primary_auditor_model'
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none',
    fp16=False,
    logging_steps=25,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
eval_metrics = trainer.evaluate()
eval_metrics

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/6327 [00:00<?, ? examples/s]

Map:   0%|          | 0/1657 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.412200,0.439640,0.814725,0.708333,0.151335,0.249389
2,0.347600,0.459027,0.802052,0.524324,0.287834,0.371648


## 6. Export structured sentence-level auditor judgments

In [ ]:
from scipy.special import softmax
import json

pred = trainer.predict(tokenized['test'])
probabilities = softmax(pred.predictions, axis=1)
predicted_labels = np.argmax(pred.predictions, axis=1)

auditor_frame = test_frame.copy()
auditor_frame['predicted_has_bias'] = predicted_labels
auditor_frame['predicted_bias_probability'] = probabilities[:, 1]
auditor_frame['prediction_confidence'] = np.max(probabilities, axis=1)

def confidence_to_label(score):
    if score < 0.2:
        return 'low'
    if score < 0.6:
        return 'medium'
    return 'high'

def rationale_for_sentence(sentence_text, label, score, bias_types):
    if label != 'biased':
        return 'The model did not find enough evidence in this sentence to support a bias judgment.'
    return f"The sentence itself is the evidence span. The model estimated {confidence_to_label(score)} bias confidence and tagged {', '.join(bias_types)} based on the wording and framing in the sentence."

records = []
for row in auditor_frame.to_dict(orient='records'):
    label = 'biased' if int(row['predicted_has_bias']) == 1 else 'not_biased'
    score = float(row['predicted_bias_probability'])
    provisional = sentence_bias_scores(score, label, row['sentence_text'])
    rationale = rationale_for_sentence(row['sentence_text'], label, score, provisional['bias_types'])
    math_scores = sentence_bias_scores(score, label, row['sentence_text'], rationale)
    record = {
        'document_id': row['article_id'],
        'event_id': row['event_id'],
        'article_id': row['article_id'],
        'source': row['source'],
        'date': row['date'],
        'title': row['title'],
        'url': row['url'],
        'main_event': row['main_event'],
        'sentence_index': int(row['sentence_index']),
        'sentence_count': int(row['sentence_count']),
        'sentence_text': row['sentence_text'],
        'judgment': {
            'bias_label': label,
            'bias_score': round(score, 6),
            'bias_strength': confidence_to_label(score),
            'bias_type': math_scores['bias_types'],
            'rationale': rationale,
            'evidence_span': row['sentence_text'],
            'math_scores': {
                's_lex': math_scores['s_lex'],
                's_inf': math_scores['s_inf'],
                'scalar_score': scalar_bias_score(math_scores),
            },
        },
        'gold_reference': {
            'has_bias': int(row['label']),
            'annotation_count': int(row['gold_annotation_count']),
            'bias_types': row['gold_bias_types'],
        },
    }
    records.append(record)

EXPORT_DIR = ROOT / 'outputs' / 'primary_auditor'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
sentence_path = EXPORT_DIR / 'auditor_sentence_judgments.jsonl'
with sentence_path.open('w') as handle:
    for record in records:
        handle.write(json.dumps(record) + '\n')

article_groups = defaultdict(list)
for record in records:
    article_groups[(record['event_id'], record['article_id'])].append(record)

article_summary = []
for (_, article_id), article_records in article_groups.items():
    sample = article_records[0]
    agg = aggregate_article_scores(article_records)
    article_summary.append({
        'article_id': article_id,
        'event_id': sample['event_id'],
        'source': sample['source'],
        'main_event': sample['main_event'],
        **agg,
    })

metrics_payload = {
    'model_name': MODEL_NAME,
    'eval_metrics': {k: float(v) for k, v in eval_metrics.items() if isinstance(v, (int, float))},
    'sentence_output': str(sentence_path),
    'document_count': len(article_summary),
}

(EXPORT_DIR / 'metrics.json').write_text(json.dumps(metrics_payload, indent=2))
pd.DataFrame(article_summary).to_csv(EXPORT_DIR / 'article_summary.csv', index=False)
print('Wrote', sentence_path)
print('Wrote', EXPORT_DIR / 'article_summary.csv')
print('Wrote', EXPORT_DIR / 'metrics.json')

## 7. Preview primary auditor outputs

In [ ]:
print((EXPORT_DIR / 'metrics.json').read_text())
with sentence_path.open() as handle:
    for _ in range(2):
        print(next(handle).strip())